# CTCF + NT motif-attention enrichment demo

This notebook organizes the known-motif enrichment workflow for the fine-tuned NT CTCF checkpoint. Heavy steps are shown as commands and are not executed by default.

In [ ]:
from pathlib import Path
import os
import sys

EXP2_ROOT = Path("/dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis")
SRC_ROOT = EXP2_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

LEGACY_MOTIF_ROOT = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif")
PEAKS_DIR = LEGACY_MOTIF_ROOT / "Data" / "original_peaks"
REFERENCE_DIR = PEAKS_DIR / "reference"
TF_NAME = "CTCF"
MODEL_TYPE = "NT"

print(f"EXP2_ROOT = {EXP2_ROOT}")
print(f"PEAKS_DIR = {PEAKS_DIR}")
print(f"REFERENCE_DIR = {REFERENCE_DIR}")

NT_CHECKPOINT = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/script/finetune/outputs/NT/motif_CTCF/checkpoint-10720")
print(f"NT_CHECKPOINT = {NT_CHECKPOINT}")
print("checkpoint exists:", NT_CHECKPOINT.exists())

In [ ]:
import pandas as pd

processed_dir = EXP2_ROOT / "Data" / "processed_data" / TF_NAME
fimo_dir = processed_dir / "fimo"
nt_dir = processed_dir / "NT"
stat_dir = EXP2_ROOT / "outputs" / "motif_attention_stats" / TF_NAME / MODEL_TYPE

pd.DataFrame({
    "artifact": ["processed_dir", "fimo_dir", "nt_mapping_dir", "stat_dir"],
    "path": [processed_dir, fimo_dir, nt_dir, stat_dir],
    "exists": [processed_dir.exists(), fimo_dir.exists(), nt_dir.exists(), stat_dir.exists()],
})

In [ ]:
# Step 1: FIMO known motif scan (requires MEME Suite / fimo).
ctcf_meme = EXP2_ROOT / "Data" / "meme" / "CTCF.meme"
positive_fasta = processed_dir / f"{TF_NAME}_positive_sequences.fasta"
fimo_command = f"""
python -m exp2_attention.fimo.extract_motif_position \\
  --input_csv {processed_dir / 'train.csv'} \\
  --input_meme {ctcf_meme} \\
  --output_dir {fimo_dir} \\
  --fasta_file {positive_fasta}

python -m exp2_attention.fimo.filter_fimo_results \\
  {fimo_dir / 'fimo.tsv'} \\
  --output {fimo_dir / 'fimo_filtered.tsv'}
"""
print(fimo_command)

In [ ]:
# Step 2: map FIMO base coordinates to NT 6-mer tokens.
mapping_command = f"""
python -m exp2_attention.mapping.map_nt_threshold \\
  --tf_name {TF_NAME} \\
  --fasta_file {positive_fasta} \\
  --fimo_file {fimo_dir / 'fimo_filtered.tsv'} \\
  --result_df_path {nt_dir / 'motif_mapping_NT_threshold.csv'} \\
  --threshold_mode
"""
print(mapping_command)

In [ ]:
# Step 3: extract NT CLS-to-token attention from the fine-tuned checkpoint.
attention_command = f"""
python -m exp2_attention.attention.extract_nt_attention \\
  --tf_name {TF_NAME} \\
  --model_path {NT_CHECKPOINT} \\
  --input_path {nt_dir / 'motif_mapping_NT_threshold.csv'} \\
  --output_path {nt_dir / 'attention_scores_NT_original.parquet'} \\
  --batch_size 64
"""
print(attention_command)

In [ ]:
# Step 4: build per-sample motif/background fold-change scores and run enrichment statistics.
stats_command = f"""
python -m exp2_attention.attention.build_attention_scores_from_parquet \\
  --tf_name {TF_NAME} \\
  --model_type NT \\
  --attention_dir {nt_dir / 'attention_scores_NT_original'} \\
  --mapping_csv {nt_dir / 'motif_mapping_NT_threshold.csv'} \\
  --output_csv {nt_dir / 'attention_scores_NT_original.csv'} \\
  --overwrite

python -m exp2_attention.stats.run_motif_attention_stats \\
  --tf_name {TF_NAME} \\
  --model_type NT \\
  --output_model_name NT \\
  --attention_path {nt_dir / 'attention_scores_NT_original'} \\
  --mapping_csv {nt_dir / 'motif_mapping_NT_threshold.csv'} \\
  --out_dir {stat_dir} \\
  --alternative greater \\
  --mode optimized
"""
print(stats_command)